# Image Vectorizer — Interactive Notebook

Convert technical diagram images (PNG / JPG) into clean **SVG vector graphics** with a transparent background.

### Workflow
1. **Run all cells** (Kernel → Restart & Run All)
2. Upload one or more diagram images with the **Upload Images** button
3. Adjust parameters in the tabs if needed *(defaults are tuned for clean CAD exports)*
4. Click **Process Images** and watch the progress bar
5. Review the **before / after** previews that appear below
6. Click **Download Results (ZIP)** to save everything

> **Tip:** Enable *Debug mode* (Cleanup tab) to save intermediate processing stages for each image.


In [ ]:
# Run this cell once to install Python dependencies.
import subprocess, sys

_pkgs = ['opencv-python', 'Pillow', 'numpy', 'ipywidgets', 'tqdm', 'matplotlib']
for _p in _pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', _p, '-q'])

# Optional: uncomment for the vtracer fallback (no external binary needed)
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'vtracer', '-q'])

print('All dependencies ready.')


In [ ]:
import io, json as _json, os, re, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import FileLink, HTML, clear_output, display
from PIL import Image

print('Imports OK.')


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Core processing functions  (self-contained, no external imports)
# ──────────────────────────────────────────────────────────────────

def _skeletonize(binary):
    """Thin all strokes to 1 px width. Tries opencv-contrib then scikit-image."""
    try:
        return cv2.ximgproc.thinning(binary, thinningType=cv2.ximgproc.THINNING_ZHANGSUEN)
    except AttributeError:
        pass
    try:
        from skimage.morphology import skeletonize as _sk
        return (_sk(binary > 0) * 255).astype(np.uint8)
    except ImportError:
        print("  [skeletonize] Requires opencv-contrib-python or scikit-image — skipping.")
        return binary


def _fix_svg_scale(svg_path, factor):
    """Rescale SVG root width/height after tracing an upscaled bitmap."""
    text = svg_path.read_text(encoding='utf-8')
    def _s(m):
        attr, val, unit = m.group(1), float(m.group(2)), m.group(3) or ''
        return f'{attr}="{val / factor:.4f}{unit}"'
    text = re.sub(r'(width)="([\d.]+)([a-z%]*)"',  _s, text, count=1)
    text = re.sub(r'(height)="([\d.]+)([a-z%]*)"', _s, text, count=1)
    svg_path.write_text(text, encoding='utf-8')


def clean_image(img_bytes, params):
    """
    Full cleaning pipeline.

    Parameters
    ----------
    img_bytes : bytes   raw image file content (PNG / JPG / BMP / TIFF)
    params    : dict    widget values

    Returns
    -------
    binary : np.ndarray  uint8, 255=line / 0=background (possibly upscaled)
    rgba   : np.ndarray  uint8 HxWx4, black lines on transparent background
    """
    arr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError('Could not decode image.')

    # Normalise to BGR
    if img.ndim == 2:
        bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.shape[2] == 4:
        bgr = img[:, :, :3]
    else:
        bgr = img.copy()

    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    # Upscale — cubic keeps anti-aliased edges intact for thin 1–3 px lines
    f = params['upscale_factor']
    if f > 1:
        h, w = gray.shape
        gray = cv2.resize(gray, (w * f, h * f), interpolation=cv2.INTER_CUBIC)

    # Median blur (for JPEG artifacts only)
    mb = params['median_blur']
    if mb > 1:
        gray = cv2.medianBlur(gray, mb | 1)

    # CLAHE — adaptive contrast for uneven scans
    if params['use_clahe']:
        clahe = cv2.createCLAHE(clipLimit=params['clahe_clip'], tileGridSize=(8, 8))
        gray  = clahe.apply(gray)

    # Threshold  →  255 = line pixel
    tt = cv2.THRESH_BINARY_INV if params['invert'] else cv2.THRESH_BINARY
    if params['use_adaptive']:
        block  = params['adaptive_block'] | 1
        binary = cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, tt, block, params['adaptive_c']
        )
    elif params['use_otsu']:
        _, binary = cv2.threshold(gray, 0, 255, tt | cv2.THRESH_OTSU)
    else:
        _, binary = cv2.threshold(gray, params['threshold'], 255, tt)

    # Morphological CLOSE — bridge tiny gaps in broken lines
    nk = params['noise_kernel']
    if nk > 0:
        k      = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (nk, nk))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k)

    # Morphological OPEN — remove isolated speckle noise
    sk = params['speckle_kernel']
    if sk > 0:
        k      = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (sk, sk))
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k)

    # Optional skeletonization
    if params['skeletonize']:
        binary = _skeletonize(binary)

    rgba = np.zeros((*binary.shape, 4), dtype=np.uint8)
    rgba[binary == 255] = [0, 0, 0, 255]
    rgba[binary == 0]   = [255, 255, 255, 0]

    return binary, rgba


def detect_text_regions(binary, upscale_factor=1):
    """
    Heuristically locate text-label components (e.g. L1, L2, L3).
    Returns list of {x, y, w, h, area} in original image coordinates.
    """
    num, _, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    boxes = []
    for i in range(1, num):
        x  = int(stats[i, cv2.CC_STAT_LEFT])
        y  = int(stats[i, cv2.CC_STAT_TOP])
        w  = int(stats[i, cv2.CC_STAT_WIDTH])
        h  = int(stats[i, cv2.CC_STAT_HEIGHT])
        ar = int(stats[i, cv2.CC_STAT_AREA])
        if not (30 <= ar <= 8000):
            continue
        aspect = w / max(h, 1)
        if not (0.15 <= aspect <= 10.0):
            continue
        if 0.85 <= aspect <= 1.15 and ar > 300:
            continue
        boxes.append({'x': x, 'y': y, 'w': w, 'h': h, 'area': ar})

    # Scale back to original image coordinates if upscaled
    if upscale_factor > 1:
        boxes = [
            {'x': b['x'] // upscale_factor, 'y': b['y'] // upscale_factor,
             'w': b['w'] // upscale_factor, 'h': b['h'] // upscale_factor,
             'area': b['area'] // (upscale_factor ** 2)}
            for b in boxes
        ]
    return boxes


def vectorize_to_svg(binary, svg_path, params):
    """Try potrace then vtracer. Returns the engine name used."""
    f = params['upscale_factor']

    # potrace
    potrace_bin = shutil.which('potrace')
    if potrace_bin:
        pbm_path = svg_path.with_suffix('.pbm')
        try:
            Image.fromarray(binary).convert('1').save(str(pbm_path))
            cmd = [
                potrace_bin, str(pbm_path),
                '--svg', '--output', str(svg_path),
                '--turdsize',     str(params['turdsize']),
                '--alphamax',     str(params['alphamax']),
                '--opttolerance', str(params['opttolerance']),
                '--resolution',   str(72 * f),
            ]
            r = subprocess.run(cmd, capture_output=True)
            if r.returncode == 0:
                return 'potrace'
        finally:
            if pbm_path.exists():
                pbm_path.unlink()

    # vtracer
    try:
        import vtracer
        tmp = svg_path.with_suffix('.tmp.png')
        try:
            cv2.imwrite(str(tmp), cv2.bitwise_not(binary))
            vtracer.convert_image_to_svg_py(
                str(tmp), str(svg_path),
                colormode='binary', hierarchical='stacked',
                filter_speckle=2, color_precision=6,
                corner_threshold=60, length_threshold=3.0,
                splice_threshold=45, path_precision=8,
            )
            if f > 1:
                _fix_svg_scale(svg_path, f)
            return 'vtracer'
        finally:
            if tmp.exists():
                tmp.unlink()
    except ImportError:
        pass

    raise RuntimeError(
        'No vectorization engine found.\n'
        'Install potrace (choco install potrace / brew install potrace) '
        'or: pip install vtracer'
    )


print('Processing functions loaded.')


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Parameter Widgets
# ──────────────────────────────────────────────────────────────────

_SL = widgets.Layout(width='340px')   # slider width

def _row(label, widget, tip=''):
    """One labelled control row with an optional tooltip."""
    lbl = widgets.HTML(
        value=f'<span title="{tip}" style="font-size:13px;cursor:help">{label}</span>',
        layout=widgets.Layout(width='220px', margin='4px 8px 0 0'),
    )
    return widgets.HBox([lbl, widget])

def _hr():
    return widgets.HTML('<hr style="margin:6px 0">')


# ── Threshold ──────────────────────────────────────────────────────
w_invert     = widgets.Checkbox(value=True,  description='Black lines on white bg', indent=False)
w_use_otsu   = widgets.Checkbox(value=True,  description='Auto threshold (Otsu)',   indent=False,
                                style={'description_width': 'initial'})
w_threshold  = widgets.IntSlider(value=200, min=50, max=254, layout=_SL,
                                  style={'description_width': '0px'})
w_use_adap   = widgets.Checkbox(value=False, description='Adaptive (uneven scans)',
                                 indent=False, style={'description_width': 'initial'})
w_adap_block = widgets.IntSlider(value=25, min=5, max=101, step=2, layout=_SL,
                                  style={'description_width': '0px'})
w_adap_c     = widgets.IntSlider(value=10, min=0, max=40, layout=_SL,
                                  style={'description_width': '0px'})
w_median     = widgets.IntSlider(value=0, min=0, max=9, step=2, layout=_SL,
                                  style={'description_width': '0px'})
w_clahe      = widgets.Checkbox(value=False, description='CLAHE contrast boost',
                                 indent=False, style={'description_width': 'initial'})
w_clahe_clip = widgets.FloatSlider(value=2.0, min=0.5, max=8.0, step=0.5, layout=_SL,
                                    style={'description_width': '0px'})

tab_threshold = widgets.VBox([
    w_invert, w_use_otsu,
    _row('Fixed threshold (0–254)', w_threshold,
         'Active only when Otsu and Adaptive are both off'),
    _hr(), w_use_adap,
    _row('Adaptive block size (odd)', w_adap_block,
         'Neighbourhood for local threshold computation'),
    _row('Adaptive C offset', w_adap_c, 'Higher = fewer pixels kept as lines'),
    _hr(),
    _row('Median blur (0=off)', w_median, 'Removes JPEG block artifacts before threshold'),
    _hr(), w_clahe,
    _row('CLAHE clip limit', w_clahe_clip, 'Contrast amplification strength'),
], layout=widgets.Layout(padding='8px'))

# ── Cleanup ────────────────────────────────────────────────────────
w_noise_k  = widgets.IntSlider(value=0, min=0, max=8, layout=_SL,
                                style={'description_width': '0px'})
w_speck_k  = widgets.IntSlider(value=0, min=0, max=6, layout=_SL,
                                style={'description_width': '0px'})
w_upscale  = widgets.Dropdown(options=[('1× (off)', 1), ('2× (recommended)', 2), ('3×', 3)],
                               value=2, layout=widgets.Layout(width='180px'))
w_skeleton = widgets.Checkbox(value=False, description='Skeletonize (thin to 1px)',
                               indent=False, style={'description_width': 'initial'})
w_debug    = widgets.Checkbox(value=False, description='Debug: save intermediate images',
                               indent=False, style={'description_width': 'initial'})

tab_cleanup = widgets.VBox([
    _row('Noise kernel (0=off)', w_noise_k,
         'Morphological CLOSE — bridges tiny gaps in broken lines'),
    _row('Speckle kernel (0=off)', w_speck_k,
         'Morphological OPEN — removes isolated dots. Keep <= line width!'),
    _row('Upscale factor', w_upscale,
         '2x gives the tracer more pixels for thin 1–3px lines; SVG size corrected automatically'),
    _hr(), w_skeleton,
    _hr(), w_debug,
], layout=widgets.Layout(padding='8px'))

# ── Vectorization ──────────────────────────────────────────────────
w_turdsize = widgets.IntSlider(value=2, min=0, max=30, layout=_SL,
                                style={'description_width': '0px'})
w_alphamax = widgets.FloatSlider(value=0.0, min=0.0, max=1.33, step=0.1,
                                  layout=_SL, readout_format='.1f',
                                  style={'description_width': '0px'})
w_opttol   = widgets.FloatSlider(value=0.1, min=0.05, max=1.0, step=0.05,
                                  layout=_SL, readout_format='.2f',
                                  style={'description_width': '0px'})

tab_vectorize = widgets.VBox([
    widgets.HTML('<b>potrace settings</b> (used when potrace CLI is installed)'),
    _row('Turd size (suppress px²)', w_turdsize,
         'Remove components smaller than N px² — keep low to preserve fine details'),
    _row('Alpha max (corner smooth)', w_alphamax,
         '0 = sharp corners (good for engineering). 1.33 = smooth curves.'),
    _row('Opt tolerance (curve fit)', w_opttol,
         'Lower = more path nodes, more faithful to original geometry'),
], layout=widgets.Layout(padding='8px'))

# ── Text detection ─────────────────────────────────────────────────
w_detect = widgets.Checkbox(value=True, description='Detect text labels (L1, L2, L3…)',
                             indent=False, style={'description_width': 'initial'})

tab_text = widgets.VBox([
    w_detect,
    widgets.HTML(
        '<p style="font-size:12px;color:#555;margin:8px 0">'
        'Detected regions are highlighted red in the preview and saved to a JSON file.<br>'
        'Use the JSON coordinates to erase labels from the SVG or replace them in PowerPoint.'
        '</p>'
    ),
], layout=widgets.Layout(padding='8px'))

# ── Tab container ──────────────────────────────────────────────────
param_tabs = widgets.Tab(children=[tab_threshold, tab_cleanup, tab_vectorize, tab_text])
for i, title in enumerate(['Threshold', 'Cleanup', 'Vectorization', 'Text Detection']):
    param_tabs.set_title(i, title)

print('Parameter widgets ready.')


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Upload, Process & Download Logic
# ──────────────────────────────────────────────────────────────────

_output_dir = Path(tempfile.mkdtemp(prefix='vectorizer_'))
_results    = []   # filled after processing

# Controls
w_upload = widgets.FileUpload(
    description='Upload Images',
    accept='.png,.jpg,.jpeg,.bmp,.tiff',
    multiple=True,
    button_style='info',
    layout=widgets.Layout(width='200px'),
)
w_run_btn = widgets.Button(
    description=' Process Images',
    button_style='success',
    icon='play',
    layout=widgets.Layout(width='180px', height='36px'),
)
w_dl_btn = widgets.Button(
    description=' Download ZIP',
    button_style='primary',
    icon='download',
    disabled=True,
    layout=widgets.Layout(width='180px', height='36px'),
)
w_progress = widgets.IntProgress(
    value=0, min=0, max=1,
    description='Idle',
    bar_style='info',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='480px'),
)
w_status  = widgets.HTML(value='<i style="color:#888">No images processed yet.</i>')
w_output  = widgets.Output()


def _collect_params():
    return {
        'invert':         w_invert.value,
        'use_otsu':       w_use_otsu.value,
        'threshold':      w_threshold.value,
        'use_adaptive':   w_use_adap.value,
        'adaptive_block': w_adap_block.value,
        'adaptive_c':     w_adap_c.value,
        'median_blur':    w_median.value,
        'use_clahe':      w_clahe.value,
        'clahe_clip':     w_clahe_clip.value,
        'noise_kernel':   w_noise_k.value,
        'speckle_kernel': w_speck_k.value,
        'upscale_factor': w_upscale.value,
        'skeletonize':    w_skeleton.value,
        'detect_text':    w_detect.value,
        'turdsize':       w_turdsize.value,
        'alphamax':       w_alphamax.value,
        'opttolerance':   w_opttol.value,
        'debug':          w_debug.value,
    }


def _preview(name, orig_bytes, rgba, text_boxes, svg_path, params):
    """Render a 3-panel before/after figure for one processed image."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(name, fontsize=12, fontweight='bold', y=1.01)

    # Panel 1 — Original
    orig_arr = np.frombuffer(orig_bytes, np.uint8)
    orig_bgr = cv2.imdecode(orig_arr, cv2.IMREAD_COLOR)
    orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
    axes[0].imshow(orig_rgb)
    axes[0].set_title('Original', fontsize=10)
    axes[0].axis('off')

    # Panel 2 — Cleaned (composite on white for display)
    white = np.full((*rgba.shape[:2], 3), 255, dtype=np.uint8)
    alpha = rgba[:, :, 3:4] / 255.0
    comp  = (rgba[:, :, :3] * alpha + white * (1 - alpha)).astype(np.uint8)
    f = params['upscale_factor']
    if f > 1:
        h, w = comp.shape[:2]
        comp = cv2.resize(comp, (w // f, h // f), interpolation=cv2.INTER_AREA)
    axes[1].imshow(comp)
    axes[1].set_title('Cleaned PNG (transparent bg)', fontsize=10)
    axes[1].axis('off')

    # Panel 3 — Original with text-box overlays (or repeat clean if none)
    if text_boxes:
        vis = orig_rgb.copy()
        for b in text_boxes:
            cv2.rectangle(vis, (b['x'], b['y']),
                          (b['x'] + b['w'], b['y'] + b['h']), (210, 40, 40), 2)
        axes[2].imshow(vis)
        axes[2].set_title(f'Detected text regions ({len(text_boxes)})', fontsize=10)
    else:
        axes[2].imshow(comp)
        axes[2].set_title('Cleaned PNG (no text detected)', fontsize=10)
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()


def _parse_uploads():
    """Handle both ipywidgets v7 (dict) and v8 (tuple) FileUpload formats."""
    val = w_upload.value
    pairs = []
    if isinstance(val, dict):
        # v7: {filename: {metadata: ..., content: bytes}}
        for name, info in val.items():
            pairs.append((info.get('metadata', {}).get('name', name), info['content']))
    else:
        # v8: tuple of FileInfo objects
        for info in val:
            pairs.append((info['name'], info['content']))
    return pairs


def on_run(btn):
    global _results
    _results = []
    w_dl_btn.disabled = True

    with w_output:
        clear_output(wait=True)

        pairs = _parse_uploads()
        if not pairs:
            display(HTML('<b style="color:#c00">No files uploaded — use the Upload Images button first.</b>'))
            return

        params = _collect_params()
        w_progress.max   = len(pairs)
        w_progress.value = 0
        w_progress.bar_style = 'info'

        for i, (name, raw) in enumerate(pairs):
            stem = Path(name).stem
            w_progress.description = f'{i + 1}/{len(pairs)}'
            w_status.value = f'<i>Processing <b>{name}</b>…</i>'

            try:
                binary, rgba = clean_image(bytes(raw), params)

                svg_path = _output_dir / f'{stem}.svg'
                engine   = vectorize_to_svg(binary, svg_path, params)

                png_path = _output_dir / f'{stem}_clean.png'
                Image.fromarray(rgba, 'RGBA').save(str(png_path))

                text_boxes = []
                if params['detect_text']:
                    text_boxes = detect_text_regions(binary, params['upscale_factor'])
                    if text_boxes:
                        (_output_dir / f'{stem}_text.json').write_text(
                            _json.dumps(text_boxes, indent=2)
                        )

                _results.append({'name': name, 'png': png_path, 'svg': svg_path, 'engine': engine})
                _preview(name, bytes(raw), rgba, text_boxes, svg_path, params)
                display(HTML(
                    f'<div style="margin-bottom:8px">'
                    f'<span style="color:green;font-weight:bold">✓ {name}</span>'
                    f' — engine: <code>{engine}</code>'
                    + (f', <b>{len(text_boxes)} text region(s)</b> saved to JSON' if text_boxes else '')
                    + '</div>'
                ))

            except Exception as exc:
                display(HTML(f'<div style="color:red"><b>✗ {name}:</b> {exc}</div>'))

            w_progress.value = i + 1

        w_progress.description = 'Done'
        w_progress.bar_style   = 'success'
        w_status.value = (
            f'<b style="color:green">✓ {len(_results)} image(s) processed.</b>'
            '  Click <em>Download ZIP</em> to save all outputs.'
        )
        w_dl_btn.disabled = (len(_results) == 0)


def on_download(btn):
    zip_path = _output_dir / 'vectorizer_results.zip'
    with zipfile.ZipFile(str(zip_path), 'w', zipfile.ZIP_DEFLATED) as zf:
        for r in _results:
            for p in [r['png'], r['svg']]:
                if p and p.exists():
                    zf.write(p, p.name)
        for jf in _output_dir.glob('*_text.json'):
            zf.write(jf, jf.name)
    try:
        from google.colab import files
        files.download(str(zip_path))
    except ImportError:
        with w_output:
            display(FileLink(str(zip_path), result_html_prefix='<b>Download: </b>'))


w_run_btn.on_click(on_run)
w_dl_btn.on_click(on_download)
print('UI callbacks registered.')


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Render the Interface
# Run this cell (or Restart & Run All) to display the full UI.
# ──────────────────────────────────────────────────────────────────

_header = widgets.HTML(
    '<h2 style="margin:0 0 2px 0">Image Vectorizer</h2>'
    '<p style="color:#555;font-size:13px;margin:0 0 10px 0">'
    'Upload diagrams → adjust parameters → process → download SVG + PNG'
    '</p>'
)

_step_upload = widgets.HTML('<b style="font-size:13px">Step 1 — Upload images</b>')
_step_params = widgets.HTML('<b style="font-size:13px">Step 2 — Adjust parameters (optional)</b>')
_step_run    = widgets.HTML('<b style="font-size:13px">Step 3 — Process & review</b>')

_btn_row  = widgets.HBox(
    [w_upload,
     widgets.HTML('<span style="margin:0 10px;line-height:36px;color:#777">then</span>'),
     w_run_btn,
     widgets.HTML('<span style="margin:0 14px;line-height:36px;color:#ccc">│</span>'),
     w_dl_btn],
)

_prog_row = widgets.HBox(
    [w_progress,
     widgets.HTML('<span style="width:10px"></span>'),
     w_status],
)

ui = widgets.VBox(
    [_header,
     widgets.HTML('<hr style="margin:0 0 10px 0">'),
     _step_upload,
     _btn_row,
     widgets.HTML('<div style="height:10px"></div>'),
     _step_params,
     param_tabs,
     widgets.HTML('<div style="height:10px"></div>'),
     _step_run,
     _prog_row,
     w_output],
    layout=widgets.Layout(padding='14px', max_width='960px'),
)

display(ui)
